In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/inbreast-dataset/INbreast Release 1.0/INbreast.csv
/kaggle/input/inbreast-dataset/INbreast Release 1.0/README.txt
/kaggle/input/inbreast-dataset/INbreast Release 1.0/inbreast.pdf
/kaggle/input/inbreast-dataset/INbreast Release 1.0/INbreast.xls
/kaggle/input/inbreast-dataset/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle ROI/53581406_muscle.roi
/kaggle/input/inbreast-dataset/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle ROI/50993616_muscle.roi
/kaggle/input/inbreast-dataset/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle ROI/22678856_muscle.roi
/kaggle/input/inbreast-dataset/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle ROI/24065680_muscle.roi
/kaggle/input/inbreast-dataset/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle ROI/24065251_muscle.roi
/kaggle/input/inbreast-dataset/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle ROI/22670147_muscle.roi
/kaggle/input/inbreast-dataset/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle ROI/22670488_musc

In [2]:
import torch
torch.cuda.empty_cache()  # Clears unused memory
torch.cuda.ipc_collect()  # Collects unreferenced tensors



In [3]:
!nvidia-smi


Thu Apr 30 00:55:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os

DATASET_DIR = "/kaggle/input/inbreast-dataset"

for item in os.listdir(DATASET_DIR):
    print(item)

INbreast Release 1.0


In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input/inbreast-dataset"):
    for d in dirs:
        if "dicom" in d.lower():
            print("DICOM folder:", os.path.join(root, d))
        if "xml" in d.lower():
            print("XML folder:", os.path.join(root, d))
    for f in files:
        if f.lower().endswith(".csv"):
            print("CSV file:", os.path.join(root, f))

XML folder: /kaggle/input/inbreast-dataset/INbreast Release 1.0/AllXML
DICOM folder: /kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs
CSV file: /kaggle/input/inbreast-dataset/INbreast Release 1.0/INbreast.csv
XML folder: /kaggle/input/inbreast-dataset/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle XML


In [6]:
DICOM_DIR = "/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs"
XML_DIR   = "/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllXML"
CSV_PATH  = "/kaggle/input/inbreast-dataset/INbreast Release 1.0/INbreast.csv"

print(os.path.exists(DICOM_DIR), DICOM_DIR)
print(os.path.exists(XML_DIR), XML_DIR)
print(os.path.exists(CSV_PATH), CSV_PATH)

True /kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs
True /kaggle/input/inbreast-dataset/INbreast Release 1.0/AllXML
True /kaggle/input/inbreast-dataset/INbreast Release 1.0/INbreast.csv


In [7]:
!pip install -q pydicom segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 46.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcugraph-cu12 24.12.0 requires pylibraft-cu12==24.12.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 24.12.0 requires rmm-cu12==24.12.*, but you have rm

In [8]:
import pydicom
import cv2
import torch
import segmentation_models_pytorch as smp

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cv2 imported")
print("smp imported")

torch: 2.5.1+cu124
cuda available: True
cv2 imported
smp imported


In [9]:
import os
import glob
import random 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader



In [10]:
#Setup path
DICOM_DIR = "/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs"
XML_DIR   = "/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllXML"
CSV_PATH  = "/kaggle/input/inbreast-dataset/INbreast Release 1.0/INbreast.csv"

print(os.path.exists(DICOM_DIR), DICOM_DIR)
print(os.path.exists(XML_DIR), XML_DIR)
print(os.path.exists(CSV_PATH), CSV_PATH)

True /kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs
True /kaggle/input/inbreast-dataset/INbreast Release 1.0/AllXML
True /kaggle/input/inbreast-dataset/INbreast Release 1.0/INbreast.csv


In [11]:
# collect files
dicom_files = sorted(glob.glob(os.path.join(DICOM_DIR, "*.dcm")))
xml_files = sorted(glob.glob(os.path.join(XML_DIR, "*.xml")))

print("DICOM files:", len(dicom_files))
print("XML files:", len(xml_files))
print("Example DICOM:", dicom_files[:2])
print("Example XML:", xml_files[:2])

DICOM files: 410
XML files: 343
Example DICOM: ['/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs/20586908_6c613a14b80a8591_MG_R_CC_ANON.dcm', '/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs/20586934_6c613a14b80a8591_MG_L_CC_ANON.dcm']
Example XML: ['/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllXML/20586908.xml', '/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllXML/20586934.xml']


In [12]:
#Read DICOM images
def read_dicom(path):
    ds = pydicom.dcmread(path)
    img = ds.pixel_array.astype(np.float32)

    img = img - img.min()
    img = img / (img.max() + 1e-8)
    img = (img * 255).astype(np.uint8)

    return img

In [13]:
# Build core table 
import pandas as pd
import os

def file_id(path):
    return os.path.splitext(os.path.basename(path))[0]

# create maps
dicom_map = {file_id(p): p for p in dicom_files}
xml_map   = {file_id(p): p for p in xml_files}

# find matching IDs
matched_ids = sorted(list(set(dicom_map.keys()) & set(xml_map.keys())))

print("Matched pairs:", len(matched_ids))

# build dataframe
data = []
for mid in matched_ids:
    data.append({
        "id": mid,
        "dicom_path": dicom_map[mid],
        "xml_path": xml_map[mid]
    })

df = pd.DataFrame(data)

df.head()

Matched pairs: 0


""


In [14]:
print([os.path.basename(p) for p in dicom_files[:10]])
print([os.path.basename(p) for p in xml_files[:10]])

['20586908_6c613a14b80a8591_MG_R_CC_ANON.dcm', '20586934_6c613a14b80a8591_MG_L_CC_ANON.dcm', '20586960_6c613a14b80a8591_MG_R_ML_ANON.dcm', '20586986_6c613a14b80a8591_MG_L_ML_ANON.dcm', '20587054_b6a4f750c6df4f90_MG_R_CC_ANON.dcm', '20587080_b6a4f750c6df4f90_MG_R_ML_ANON.dcm', '20587148_fd746d25eb40b3dc_MG_R_CC_ANON.dcm', '20587174_fd746d25eb40b3dc_MG_L_CC_ANON.dcm', '20587200_fd746d25eb40b3dc_MG_R_ML_ANON.dcm', '20587226_fd746d25eb40b3dc_MG_L_ML_ANON.dcm']
['20586908.xml', '20586934.xml', '20586960.xml', '20586986.xml', '20587054.xml', '20587080.xml', '20587148.xml', '20587174.xml', '20587200.xml', '20587226.xml']


In [15]:
# Match only the first number before _
import os, re, glob
import pandas as pd

dicom_files = sorted(glob.glob(os.path.join(DICOM_DIR, "*.dcm")))
xml_files   = sorted(glob.glob(os.path.join(XML_DIR, "*.xml")))

def dicom_id(path):
    name = os.path.basename(path)
    return name.split("_")[0]   # 20586908

def xml_id(path):
    name = os.path.basename(path)
    return os.path.splitext(name)[0]  # 20586908

dicom_map = {}
for p in dicom_files:
    did = dicom_id(p)
    dicom_map.setdefault(did, []).append(p)

xml_map = {xml_id(p): p for p in xml_files}

matched_ids = sorted(set(dicom_map.keys()) & set(xml_map.keys()))

print("DICOM IDs:", len(dicom_map))
print("XML IDs:", len(xml_map))
print("Matched IDs:", len(matched_ids))
print(matched_ids[:10])

DICOM IDs: 410
XML IDs: 343
Matched IDs: 343
['20586908', '20586934', '20586960', '20586986', '20587054', '20587080', '20587148', '20587174', '20587200', '20587226']


In [16]:
#Build core df
data = []

for mid in matched_ids:
    for dicom_path in dicom_map[mid]:
        data.append({
            "id": mid,
            "dicom_path": dicom_path,
            "xml_path": xml_map[mid]
        })

df = pd.DataFrame(data)

print(df.shape)
df.head()

(343, 3)


,id,dicom_path,xml_path
0,20586908,/kaggle/input/inbreast-dataset/INbreast Releas...,/kaggle/input/inbreast-dataset/INbreast Releas...
1,20586934,/kaggle/input/inbreast-dataset/INbreast Releas...,/kaggle/input/inbreast-dataset/INbreast Releas...
2,20586960,/kaggle/input/inbreast-dataset/INbreast Releas...,/kaggle/input/inbreast-dataset/INbreast Releas...
3,20586986,/kaggle/input/inbreast-dataset/INbreast Releas...,/kaggle/input/inbreast-dataset/INbreast Releas...
4,20587054,/kaggle/input/inbreast-dataset/INbreast Releas...,/kaggle/input/inbreast-dataset/INbreast Releas...


In [17]:
# Read XML countor and make masks

def extract_points_from_xml(xml_path):
    with open(xml_path, "rb") as f:
        data = plistlib.load(f)

    points_list = []

    try:
        rois = data["Images"][0]["ROIs"]
    except:
        return points_list

    for roi in rois:
        if "Point_px" in roi:
            points = roi["Point_px"]
            contour = []

            for p in points:
                p = p.replace("(", "").replace(")", "")
                x, y = p.split(",")
                contour.append([float(x), float(y)])

            if len(contour) > 2:
                points_list.append(np.array(contour, dtype=np.int32))

    return points_list


def xml_to_mask(xml_path, image_shape):
    mask = np.zeros(image_shape, dtype=np.uint8)
    contours = extract_points_from_xml(xml_path)

    for contour in contours:
        cv2.fillPoly(mask, [contour], 1)

    return mask

In [18]:
row = df.sample(1).iloc[0]

img = read_dicom(row["dicom_path"])
xml_path = row["xml_path"]

print("DICOM:", row["dicom_path"])
print("XML:", xml_path)

with open(xml_path, "rb") as f:
    data = plistlib.load(f)

print(data.keys())
print(data["Images"][0].keys())
print("Number of ROIs:", len(data["Images"][0]["ROIs"]))

for i, roi in enumerate(data["Images"][0]["ROIs"]):
    print("\nROI", i)
    print(roi.keys())
    print("Name:", roi.get("Name"))
    print("Number of points:", len(roi.get("Point_px", [])))
    print("First points:", roi.get("Point_px", [])[:5])

DICOM: /kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs/53586869_6ac23356b912ee9b_MG_L_ML_ANON.dcm
XML: /kaggle/input/inbreast-dataset/INbreast Release 1.0/AllXML/53586869.xml


NameError: name 'plistlib' is not defined

In [ ]:
#Replace XML function
def extract_points_from_xml(xml_path):
    with open(xml_path, "rb") as f:
        data = plistlib.load(f)

    points_list = []

    for image in data.get("Images", []):
        for roi in image.get("ROIs", []):
            points = roi.get("Point_px", [])

            contour = []
            for p in points:
                if isinstance(p, str):
                    p = p.replace("(", "").replace(")", "")
                    x, y = p.split(",")
                    contour.append([float(x), float(y)])

            if len(contour) > 2:
                points_list.append(np.array(contour, dtype=np.int32))

    return points_list


def xml_to_mask(xml_path, image_shape):
    mask = np.zeros(image_shape, dtype=np.uint8)
    contours = extract_points_from_xml(xml_path)

    print("Contours found:", len(contours))

    for contour in contours:
        cv2.fillPoly(mask, [contour], 1)

    print("Mask unique:", np.unique(mask))

    return mask

In [ ]:
show_random_image_mask_pair(df)

In [ ]:
#Pytorch dataset
IMG_SIZE = 512

class INBreastSegDataset(Dataset):
    def __init__(self, dataframe, img_size=512):
        self.df = dataframe.reset_index(drop=True)
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        img = read_dicom(row["dicom_path"])
        mask = xml_to_mask(row["xml_path"], img.shape)

        img = cv2.resize(img, (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        img = img.astype(np.float32) / 255.0
        mask = mask.astype(np.float32)

        img = np.expand_dims(img, axis=0)
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

In [ ]:
#Train-validation Split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = INBreastSegDataset(train_df, IMG_SIZE)
val_dataset = INBreastSegDataset(val_df, IMG_SIZE)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2)

print(len(train_dataset), len(val_dataset))

In [ ]:
#Build ResNet50 - UNet model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=1,
    classes=1,
    activation=None
)

model = model.to(device)

In [ ]:
#Loss and matrics
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)

        intersection = (probs * targets).sum()
        dice = (2 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)

        return 1 - dice


bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()

def combined_loss(logits, masks):
    return bce_loss(logits, masks) + dice_loss(logits, masks)


def dice_score(logits, masks, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    intersection = (preds * masks).sum()
    union = preds.sum() + masks.sum()

    dice = (2 * intersection + 1e-6) / (union + 1e-6)
    return dice.item()


def iou_score(logits, masks, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    intersection = (preds * masks).sum()
    union = preds.sum() + masks.sum() - intersection

    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.item()



In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = combined_loss(logits, masks)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0
    val_dice = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = combined_loss(logits, masks)

            val_loss += loss.item()
            val_dice += dice_score(logits, masks)
            val_iou += iou_score(logits, masks)

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    val_iou /= len(val_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Dice: {val_dice:.4f} | "
        f"IoU: {val_iou:.4f}"
    )

In [ ]:
import torch
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=1,
    classes=1,
    activation=None
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Model and optimizer are defined.")

In [ ]:
CHECKPOINT_PATH = "/kaggle/working/resnet50_unet_checkpoint.pth"

torch.save({
    "epoch": 0,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, CHECKPOINT_PATH)

print("Checkpoint saved.")

In [ ]:
print(model)
print(optimizer)

In [ ]:
# Train ResNet 50 -Unet segmentation model using 50 epochs
# import
import os, glob, random, plistlib, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pydicom
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

DICOM_DIR = "/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs"
XML_DIR   = "/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllXML"

WORK_DIR = "/kaggle/working"

IMG_SIZE_SEG = 256
IMG_SIZE_CLS = 224
BATCH_SIZE = 2
EPOCHS_SEG = 50
EPOCHS_CLS = 20

In [ ]:
# Load the 4 saved csvs
csv_files = [
    "/kaggle/working/CC_normal_C_D.csv",
    "/kaggle/working/CC_cancer_C_D.csv",
    "/kaggle/working/MLO_normal_C_D.csv",
    "/kaggle/working/MLO_cancer_C_D.csv"
]

df_csv = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

df_csv["File Name"] = df_csv["File Name"].astype(str)
df_csv["label"] = df_csv["label"].astype(int)

print(df_csv.shape)
print(df_csv["label"].value_counts())
display(df_csv.head())

In [ ]:
# Build DICOM & XML paired dataframe
dicom_files = sorted(glob.glob(os.path.join(DICOM_DIR, "*.dcm")))
xml_files = sorted(glob.glob(os.path.join(XML_DIR, "*.xml")))

def dicom_id(path):
    return os.path.basename(path).split("_")[0]

def xml_id(path):
    return os.path.splitext(os.path.basename(path))[0]

dicom_map = {}
for p in dicom_files:
    did = dicom_id(p)
    dicom_map.setdefault(did, []).append(p)

xml_map = {xml_id(p): p for p in xml_files}

matched_ids = sorted(set(dicom_map.keys()) & set(xml_map.keys()) & set(df_csv["File Name"]))

data = []
for mid in matched_ids:
    label_row = df_csv[df_csv["File Name"] == mid].iloc[0]
    for dicom_path in dicom_map[mid]:
        data.append({
            "id": mid,
            "dicom_path": dicom_path,
            "xml_path": xml_map[mid],
            "label": int(label_row["label"]),
            "View": label_row["View"],
            "Laterality": label_row["Laterality"],
            "ACR": label_row["ACR"],
            "Bi-Rads": label_row["Bi-Rads"],
            "density_domain": label_row["density_domain"]
        })

df_final = pd.DataFrame(data)

print(df_final.shape)
print(df_final["label"].value_counts())
display(df_final.head())

In [ ]:
# DICOM & XML Mask Function
def read_dicom(path):
    ds = pydicom.dcmread(path)
    img = ds.pixel_array.astype(np.float32)
    img = img - img.min()
    img = img / (img.max() + 1e-8)
    img = (img * 255).astype(np.uint8)
    return img


def extract_points_from_xml(xml_path):
    with open(xml_path, "rb") as f:
        data = plistlib.load(f)

    points_list = []

    for image in data.get("Images", []):
        for roi in image.get("ROIs", []):
            points = roi.get("Point_px", [])
            contour = []

            for p in points:
                if isinstance(p, str):
                    p = p.replace("(", "").replace(")", "")
                    x, y = p.split(",")
                    contour.append([float(x), float(y)])

            if len(contour) > 2:
                points_list.append(np.array(contour, dtype=np.int32))

    return points_list


def xml_to_mask(xml_path, image_shape):
    mask = np.zeros(image_shape, dtype=np.uint8)
    contours = extract_points_from_xml(xml_path)

    for contour in contours:
        cv2.fillPoly(mask, [contour], 1)

    return mask

In [ ]:
# Visual sanity check
row = df_final.sample(1).iloc[0]

img = read_dicom(row["dicom_path"])
mask = xml_to_mask(row["xml_path"], img.shape)

print("ID:", row["id"])
print("Label:", row["label"])
print("Mask values:", np.unique(mask))

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img, cmap="gray")
plt.title("DICOM image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask, cmap="gray")
plt.title("Ground-truth mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(img, cmap="gray")
plt.imshow(mask, alpha=0.4, cmap="Reds")
plt.title("Overlay")
plt.axis("off")

plt.show()

In [ ]:
class INBreastSegDataset(Dataset):
    def __init__(self, dataframe, img_size=256):
        self.df = dataframe.reset_index(drop=True)
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        img = read_dicom(row["dicom_path"])
        mask = xml_to_mask(row["xml_path"], img.shape)

        img = cv2.resize(img, (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        img = img.astype(np.float32) / 255.0
        mask = mask.astype(np.float32)

        img = np.expand_dims(img, 0)
        mask = np.expand_dims(mask, 0)

        return torch.tensor(img), torch.tensor(mask)

In [ ]:
train_df, temp_df = train_test_split(
    df_final,
    test_size=0.3,
    random_state=SEED,
    stratify=df_final["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_seg_ds = INBreastSegDataset(train_df, IMG_SIZE_SEG)
val_seg_ds = INBreastSegDataset(val_df, IMG_SIZE_SEG)
test_seg_ds = INBreastSegDataset(test_df, IMG_SIZE_SEG)

train_seg_loader = DataLoader(train_seg_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_seg_loader = DataLoader(val_seg_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_seg_loader = DataLoader(test_seg_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(len(train_seg_ds), len(val_seg_ds), len(test_seg_ds))

In [ ]:
# TransUNet + Pixel Level contrastive learning
class TransUNetContrastive(nn.Module):
    def __init__(self):
        super().__init__()

        self.segmenter = smp.Unet(
            encoder_name="resnet50",
            encoder_weights="imagenet",
            in_channels=1,
            classes=1,
            activation=None
        )

        self.embedding_head = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, kernel_size=1)
        )

    def forward(self, x):
        logits = self.segmenter(x)
        embeddings = self.embedding_head(logits)
        return logits, embeddings

In [ ]:
# Losses + Matrics
bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0]).to(device))

def dice_loss(logits, masks, smooth=1e-6):
    probs = torch.sigmoid(logits)
    intersection = (probs * masks).sum()
    dice = (2 * intersection + smooth) / (probs.sum() + masks.sum() + smooth)
    return 1 - dice

def segmentation_loss(logits, masks):
    return 0.5 * bce_loss(logits, masks) + 0.5 * dice_loss(logits, masks)

def pixel_contrastive_loss(embeddings, masks, temperature=0.1, max_pixels=2048):
    B, C, H, W = embeddings.shape

    emb = embeddings.permute(0, 2, 3, 1).reshape(-1, C)
    lbl = masks.reshape(-1)

    idx = torch.randperm(emb.shape[0], device=emb.device)[:max_pixels]
    emb = emb[idx]
    lbl = lbl[idx]

    emb = nn.functional.normalize(emb, dim=1)
    sim = torch.matmul(emb, emb.T) / temperature

    lbl = lbl.unsqueeze(1)
    pos_mask = (lbl == lbl.T).float()

    exp_sim = torch.exp(sim)
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

    loss = -(pos_mask * log_prob).sum() / (pos_mask.sum() + 1e-8)
    return loss

def dice_score(logits, masks, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * masks).sum()
    return ((2 * intersection) / (preds.sum() + masks.sum() + 1e-6)).item()

def iou_score(logits, masks, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * masks).sum()
    union = preds.sum() + masks.sum() - intersection
    return (intersection / (union + 1e-6)).item()

In [ ]:
# -------------------------------
# Training segmentation model for 50 epochs
# -------------------------------
print("Train batches:", len(train_seg_loader))
print("Val batches:", len(val_seg_loader))

images, masks = next(iter(train_seg_loader))
print("Image shape:", images.shape)
print("Mask shape:", masks.shape)
print("Image min/max:", images.min().item(), images.max().item())
print("Mask unique:", torch.unique(masks))
print("Mask sum:", masks.sum().item())

In [ ]:
seg_model = TransUNetContrastive().to(device)
optimizer_seg = torch.optim.Adam(seg_model.parameters(), lr=1e-4)

LAMBDA_CONTRAST = 0.1
best_val_dice = 0.0
history_seg = []

CHECKPOINT_PATH = "/kaggle/working/transunet_contrastive_checkpoint.pth"
BEST_PATH = "/kaggle/working/best_transunet_contrastive.pth"

for epoch in range(EPOCHS_SEG):
    seg_model.train()
    train_loss = 0.0
    train_seg_loss = 0.0
    train_con_loss = 0.0

    for batch_idx, (images, masks) in enumerate(train_seg_loader):
        images = images.float().to(device)
        masks = masks.float().to(device)

        optimizer_seg.zero_grad()

        logits, embeddings = seg_model(images)

        loss_seg = segmentation_loss(logits, masks)

        # Only use contrastive loss if mask has positive pixels
        if masks.sum() > 0:
            loss_con = pixel_contrastive_loss(embeddings, masks)
        else:
            loss_con = torch.tensor(0.0, device=device)

        loss = loss_seg + LAMBDA_CONTRAST * loss_con

        if torch.isnan(loss):
            print("NaN loss detected. Skipping batch:", batch_idx)
            continue

        loss.backward()
        optimizer_seg.step()

        train_loss += loss.item()
        train_seg_loss += loss_seg.item()
        train_con_loss += loss_con.item()

    train_loss /= max(len(train_seg_loader), 1)
    train_seg_loss /= max(len(train_seg_loader), 1)
    train_con_loss /= max(len(train_seg_loader), 1)

    # -------------------------------
    # Validation
    # -------------------------------
    seg_model.eval()
    val_dice = 0.0
    val_iou = 0.0

    with torch.no_grad():
        for images, masks in val_seg_loader:
            images = images.float().to(device)
            masks = masks.float().to(device)

            logits, _ = seg_model(images)

            val_dice += dice_score(logits, masks)
            val_iou += iou_score(logits, masks)

    val_dice /= max(len(val_seg_loader), 1)
    val_iou /= max(len(val_seg_loader), 1)

    history_seg.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_seg_loss": train_seg_loss,
        "train_contrast_loss": train_con_loss,
        "val_dice": val_dice,
        "val_iou": val_iou
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS_SEG} | "
        f"Loss: {train_loss:.4f} | "
        f"Seg: {train_seg_loss:.4f} | "
        f"Contrast: {train_con_loss:.4f} | "
        f"Dice: {val_dice:.4f} | "
        f"IoU: {val_iou:.4f}"
    )

    # Save every epoch
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": seg_model.state_dict(),
        "optimizer_state_dict": optimizer_seg.state_dict(),
        "best_val_dice": best_val_dice,
        "history": history_seg
    }, CHECKPOINT_PATH)

    # Save best model
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(seg_model.state_dict(), BEST_PATH)
        print("Best model saved.")

pd.DataFrame(history_seg).to_csv(
    "/kaggle/working/segmentation_history.csv",
    index=False
)

print("Training finished.")

In [ ]:
# Save epoch history
seg_history_path = "/kaggle/working/segmentation_history.csv"
hist_df = pd.DataFrame(history_seg)
hist_df.to_csv(seg_history_path, index=False)

print("Saved:", seg_history_path)
display(hist_df.head())
display(hist_df.tail())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

hist_df = pd.read_csv("/kaggle/working/segmentation_history.csv")

plt.figure(figsize=(8, 5))
plt.plot(hist_df["epoch"], hist_df["train_loss"], label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Segmentation Training Loss over 50 Epochs")
plt.legend()
plt.grid(True)
plt.savefig("/kaggle/working/segmentation_train_loss.png", dpi=300)
plt.show()

In [ ]:
# Evaluate Segmentation model on test set
seg_model.load_state_dict(torch.load("/kaggle/working/best_transunet_contrastive.pth", map_location=device))
seg_model.eval()

test_dice = 0
test_iou = 0

with torch.no_grad():
    for images, masks in test_seg_loader:
        images = images.float().to(device)
        masks = masks.float().to(device)

        logits, _ = seg_model(images)

        test_dice += dice_score(logits, masks)
        test_iou += iou_score(logits, masks)

test_dice /= len(test_seg_loader)
test_iou /= len(test_seg_loader)

print("Test Dice:", test_dice)
print("Test IoU:", test_iou)

In [ ]:
# DensNet201 ROI Classification
def crop_roi_from_mask(image, mask):
    coords = np.argwhere(mask > 0)

    if coords.shape[0] == 0:
        return image

    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)

    pad = 20
    y_min = max(y_min - pad, 0)
    x_min = max(x_min - pad, 0)
    y_max = min(y_max + pad, image.shape[0] - 1)
    x_max = min(x_max + pad, image.shape[1] - 1)

    return image[y_min:y_max+1, x_min:x_max+1]


class INBreastROIDataset(Dataset):
    def __init__(self, dataframe, seg_model, img_size=224):
        self.df = dataframe.reset_index(drop=True)
        self.seg_model = seg_model.eval()
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        img = read_dicom(row["dicom_path"])
        img_resized = cv2.resize(img, (IMG_SIZE_SEG, IMG_SIZE_SEG))

        x = img_resized.astype(np.float32) / 255.0
        x_tensor = torch.tensor(x).unsqueeze(0).unsqueeze(0).float().to(device)

        with torch.no_grad():
            logits, _ = self.seg_model(x_tensor)
            pred = torch.sigmoid(logits)[0, 0].cpu().numpy()

        pred_mask = (pred > 0.5).astype(np.uint8)

        roi = crop_roi_from_mask(x, pred_mask)
        roi = cv2.resize(roi, (self.img_size, self.img_size))
        roi = np.stack([roi, roi, roi], axis=0)

        label = int(row["label"])

        return torch.tensor(roi, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [ ]:
# Densnet201 Classifier
train_cls_ds = INBreastROIDataset(train_df, seg_model, IMG_SIZE_CLS)
val_cls_ds = INBreastROIDataset(val_df, seg_model, IMG_SIZE_CLS)
test_cls_ds = INBreastROIDataset(test_df, seg_model, IMG_SIZE_CLS)

train_cls_loader = DataLoader(train_cls_ds, batch_size=4, shuffle=True, num_workers=0)
val_cls_loader = DataLoader(val_cls_ds, batch_size=4, shuffle=False, num_workers=0)
test_cls_loader = DataLoader(test_cls_ds, batch_size=4, shuffle=False, num_workers=0)

In [ ]:
densenet = models.densenet201(weights="IMAGENET1K_V1")
num_features = densenet.classifier.in_features
densenet.classifier = nn.Linear(num_features, 2)
densenet = densenet.to(device)

criterion_cls = nn.CrossEntropyLoss()
optimizer_cls = torch.optim.Adam(densenet.parameters(), lr=1e-4)

best_val_acc = 0
history_cls = []

In [ ]:
# Train DesNet201
for epoch in range(EPOCHS_CLS):
    densenet.train()
    train_loss = 0

    for images, labels in train_cls_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_cls.zero_grad()

        outputs = densenet(images)
        loss = criterion_cls(outputs, labels)

        loss.backward()
        optimizer_cls.step()

        train_loss += loss.item()

    densenet.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_cls_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = densenet(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    train_loss /= len(train_cls_loader)

    history_cls.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_acc": val_acc
    })

    print(f"Epoch {epoch+1}/{EPOCHS_CLS} | Loss {train_loss:.4f} | Val Acc {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(densenet.state_dict(), "/kaggle/working/best_densenet201_roi.pth")

pd.DataFrame(history_cls).to_csv("/kaggle/working/classification_history.csv", index=False)

In [ ]:
# Classification evaluation
densenet.load_state_dict(torch.load("/kaggle/working/best_densenet201_roi.pth", map_location=device))
densenet.eval()

all_probs = []
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_cls_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = densenet(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("ROC-AUC:", roc_auc_score(all_labels, all_probs))
print(classification_report(all_labels, all_preds, target_names=["Normal/Benign-like", "Suspicious/Malignant-like"]))

cm = confusion_matrix(all_labels, all_preds)
print(cm)

In [ ]:
cls_hist = pd.read_csv("/kaggle/working/classification_history.csv")

plt.figure(figsize=(7, 5))
plt.plot(cls_hist["epoch"], cls_hist["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("DenseNet201 ROI Classification Performance")
plt.legend()
plt.grid()
plt.savefig("/kaggle/working/classification_metrics.png", dpi=300)
plt.show()

In [ ]:
torch.save({
    "epoch": epoch + 1,
    "model_state_dict": seg_model.state_dict(),
    "optimizer_state_dict": optimizer_seg.state_dict(),
    "best_val_dice": best_val_dice,
    "history_seg": history_seg
}, "/kaggle/working/transunet_latest_checkpoint.pth")

pd.DataFrame(history_seg).to_csv(
    "/kaggle/working/segmentation_history.csv",
    index=False
)

In [ ]:
#save the best model
torch.save(
    seg_model.state_dict(),
    "/kaggle/working/best_transunet_contrastive.pth"
)

In [ ]:
#Save Version → Quick Save

In [ ]:
# import pandas as pd

# # Load the dataset
# csv_path = "/kaggle/input/will-modif/breast-level_annotations.csv"
# df = pd.read_csv(csv_path)

# # Convert BI-RADS 0 → 1
# # df["breast_birads"] = df["breast_birads"].replace("BI-RADS 0", "BI-RADS 1")

# # Assign labels based on BI-RADS categories
# def assign_label(birads):
#     if birads in ["BI-RADS 1",]:
#         return 0  # Benign
#     elif birads in ["BI-RADS 2","BI-RADS 3", "BI-RADS 4", "BI-RADS 5"]:
#         return 1  # Suspicious/Malignant
#     return None  # Handle missing or unknown values

# df["label"] = df["breast_birads"].apply(assign_label)

# # Keep only BI-RADS 1, 2, 3, 4, 5
# df = df[df["breast_birads"].isin(["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"])]

# Remove cases where (BI-RADS is 1 or 2) AND (Density is D)
# # Remove all cases where breast_density is "DENSITY D"
# df = df[df["breast_density"] != "DENSITY D"]


# # Ensure each study has exactly CC & MLO views
# study_counts = df.groupby("study_id")["view_position"].nunique()
# valid_studies = study_counts[study_counts == 2].index  # Expecting CC & MLO views
# df = df[df["study_id"].isin(valid_studies)]

# # Save the cleaned dataset
# filtered_csv_path = "result.csv"
# df.to_csv(filtered_csv_path, index=False)

# print("Filtered dataset saved to:", filtered_csv_path)
# print(df.head())


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith((".csv", ".dcm", ".xml")):
            print(os.path.join(root, f))

In [ ]:
# Load dataset
import pandas as pd

csv_path = "/kaggle/input/inbreast-dataset/INbreast Release 1.0/INbreast.csv"

df_raw = pd.read_csv(csv_path, sep=";")

print(df_raw.shape)
print(df_raw.columns)
display(df_raw.head(3))
df_csv = pd.read_csv(csv_path, sep=";")

print(df_csv.shape)
print(df_csv.columns)
df_csv.head()

In [ ]:
df_csv = df_raw.copy()

# --- Clean columns ---
df_csv["ACR"] = (
    df_csv["ACR"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

df_csv["Bi-Rads"] = (
    df_csv["Bi-Rads"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

df_csv["View"] = (
    df_csv["View"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# --- Drop rows with missing key info early ---
df_csv = df_csv.dropna(subset=["ACR", "Bi-Rads", "View"]).copy()

# --- Filter density (C, D only) ---
df_csv = df_csv[df_csv["ACR"].isin([3, 4])].copy()
df_csv["density_domain"] = df_csv["ACR"].map({3: "C", 4: "D"})

# --- Filter valid BI-RADS ---
df_csv = df_csv[df_csv["Bi-Rads"].isin([1, 2, 3, 4, 5, 6])].copy()

# --- Assign binary labels ---
df_csv["label"] = (df_csv["Bi-Rads"] >= 4).astype(int)

# --- Keep only CC / MLO views ---
df_csv = df_csv[df_csv["View"].isin(["CC", "MLO"])].copy()

# --- Keep only necessary columns ---
df_csv = df_csv.loc[:, [
    "File Name", "Laterality", "View",
    "ACR", "Bi-Rads", "density_domain", "label"
]].copy()

# --- Reset index (cleaner downstream use) ---
df_csv.reset_index(drop=True, inplace=True)

# --- Debug prints ---
print("Final shape:", df_csv.shape)
print("Label distribution:\n", df_csv["label"].value_counts())

display(df_csv.head(2))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

def split_and_save(csv_path, prefix, test_size=0.15, val_size=0.15):
    df_tmp = pd.read_csv(csv_path)

    train_df, test_df = train_test_split(
        df_tmp,
        test_size=test_size,
        random_state=42,
        stratify=df_tmp["label"] if "label" in df_tmp.columns else None
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=val_size,
        random_state=42,
        stratify=train_df["label"] if "label" in train_df.columns else None
    )

    train_df.to_csv(f"/kaggle/working/{prefix}_train.csv", index=False)
    val_df.to_csv(f"/kaggle/working/{prefix}_val.csv", index=False)
    test_df.to_csv(f"/kaggle/working/{prefix}_test.csv", index=False)

    print(prefix)
    print("Train:", train_df.shape)
    print("Val:", val_df.shape)
    print("Test:", test_df.shape)
    print("-" * 50)

In [ ]:
df_csv[(df_csv["View"] == "CC") & (df_csv["label"] == 0)].to_csv("/kaggle/working/CC_normal_C_D.csv", index=False)
df_csv[(df_csv["View"] == "CC") & (df_csv["label"] == 1)].to_csv("/kaggle/working/CC_cancer_C_D.csv", index=False)
df_csv[(df_csv["View"] == "MLO") & (df_csv["label"] == 0)].to_csv("/kaggle/working/MLO_normal_C_D.csv", index=False)
df_csv[(df_csv["View"] == "MLO") & (df_csv["label"] == 1)].to_csv("/kaggle/working/MLO_cancer_C_D.csv", index=False)

print("Saved 4 CSV files.")

In [ ]:
import os

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

groups = {
    "CC_normal_C_D.csv":  (df_csv["View"].eq("CC"))  & (df_csv["label"].eq(0)),
    "CC_cancer_C_D.csv":  (df_csv["View"].eq("CC"))  & (df_csv["label"].eq(1)),
    "MLO_normal_C_D.csv": (df_csv["View"].eq("MLO")) & (df_csv["label"].eq(0)),
    "MLO_cancer_C_D.csv": (df_csv["View"].eq("MLO")) & (df_csv["label"].eq(1)),
}

for filename, condition in groups.items():
    subset = df_csv[condition].copy()
    save_path = os.path.join(OUT_DIR, filename)
    subset.to_csv(save_path, index=False)
    print(f"Saved {filename}: {subset.shape[0]} rows")

print("Saved 4 CSV files to /kaggle/working")

In [ ]:
for f in os.listdir("/kaggle/working"):
    if f.endswith(".csv"):
        print(f)

In [ ]:
import os

WORK_DIR = "/kaggle/working"

cc_cancer_csv  = os.path.join(WORK_DIR, "CC_cancer_C_D.csv")
cc_normal_csv  = os.path.join(WORK_DIR, "CC_normal_C_D.csv")
mlo_cancer_csv = os.path.join(WORK_DIR, "MLO_cancer_C_D.csv")
mlo_normal_csv = os.path.join(WORK_DIR, "MLO_normal_C_D.csv")

print(os.path.exists(cc_cancer_csv), cc_cancer_csv)
print(os.path.exists(cc_normal_csv), cc_normal_csv)
print(os.path.exists(mlo_cancer_csv), mlo_cancer_csv)
print(os.path.exists(mlo_normal_csv), mlo_normal_csv)

In [ ]:
#Split data
from sklearn.model_selection import train_test_split
import pandas as pd

def split_and_save(csv_path, prefix):
    df_tmp = pd.read_csv(csv_path)

    train_df, test_df = train_test_split(
        df_tmp,
        test_size=0.15,
        random_state=42
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=0.15,
        random_state=42
    )

    train_df.to_csv(f"/kaggle/working/{prefix}_train.csv", index=False)
    val_df.to_csv(f"/kaggle/working/{prefix}_val.csv", index=False)
    test_df.to_csv(f"/kaggle/working/{prefix}_test.csv", index=False)

    print(prefix, train_df.shape, val_df.shape, test_df.shape)

split_and_save(cc_cancer_csv, "CC_cancer")
split_and_save(cc_normal_csv, "CC_normal")
split_and_save(mlo_cancer_csv, "MLO_cancer")
split_and_save(mlo_normal_csv, "MLO_normal")

In [ ]:
train_csv = "/kaggle/working/CC_cancer_train.csv"
test_csv  = "/kaggle/working/CC_cancer_test.csv"

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from torchvision import transforms
from itertools import cycle


In [ ]:
#Find original PNG folder
import os

for root, dirs, files in os.walk("/kaggle/input"):
    png_files = [f for f in files if f.lower().endswith(".png")]
    if len(png_files) > 0:
        print("PNG folder:", root)
        print("Example files:", png_files[:5])
        print("-" * 60)

In [ ]:
root_dir = "/kaggle/input/inbreast-png-dataset/Inbreast_png"

In [ ]:
print("Image root exists:", os.path.exists(root_dir), root_dir)

In [ ]:
class SingleImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.samples = dataframe.to_dict(orient="records")
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row = self.samples[idx]

        image_path = os.path.join(
            self.root_dir,
            row["study_id"],
            row["image_id"] + ".png"
        )

        if not os.path.exists(image_path):
            return self.__getitem__((idx + 1) % len(self.samples))

        image = Image.open(image_path).convert("L")

        # Flip right breast for consistency
        if row["laterality"] == "R":
            image = image.transpose(Image.FLIP_LEFT_RIGHT)

        if self.transform:
            image = self.transform(image)

        return image
def get_cycle_gan_datasets(csv_path, root_dir, transform):
    df = pd.read_csv(csv_path)

    df_C = df[df["density_domain"] == "C"].reset_index(drop=True)
    df_D = df[df["density_domain"] == "D"].reset_index(drop=True)

    dataset_C = SingleImageDataset(df_C, root_dir, transform)
    dataset_D = SingleImageDataset(df_D, root_dir, transform)

    return dataset_C, dataset_D
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])
root_dir = "/kaggle/input/research/NewPngImages - Copy"

train_csv = "CC_cancer_train.csv"
test_csv  = "CC_cancer_test.csv"

train_C, train_D = get_cycle_gan_datasets(train_csv, root_dir, train_transform)
test_C,  test_D  = get_cycle_gan_datasets(test_csv, root_dir, val_test_transform)
batch_size = 4

# Density C loader (normal shuffle)
train_C_loader = DataLoader(
    train_C,
    batch_size=batch_size,
    shuffle=True
)

# Density D loader (oversampled)
sampler_D = WeightedRandomSampler(
    weights=[1.0] * len(train_D),
    num_samples=len(train_C),
    replacement=True
)

train_D_loader = DataLoader(
    train_D,
    batch_size=batch_size,
    sampler=sampler_D
)


In [ ]:
import torch
import torch.nn as nn

# -------------------------------
# 🔹 CBAM Attention Module
# -------------------------------
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out) * x


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        assert kernel_size in (3, 7), "Kernel size must be 3 or 7"
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        attn = self.sigmoid(self.conv(x_cat))
        return attn * x


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


# -------------------------------
# 🔹 ResNet Block with CBAM
# -------------------------------
class ResnetBlock(nn.Module):
    def __init__(self, dim, use_cbam=False):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=0),
            nn.InstanceNorm2d(dim),
            nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=0),
            nn.InstanceNorm2d(dim)
        )
        self.use_cbam = use_cbam
        if use_cbam:
            self.attn = CBAM(dim)

    def forward(self, x):
        out = self.block(x)
        if self.use_cbam:
            out = self.attn(out)  # Apply CBAM attention
        return x + out  # Residual connection


# -------------------------------
# 🔹 ResNet Generator with Attention
# -------------------------------
class ResnetGenerator(nn.Module):
    def __init__(self, input_nc=3, output_nc=3, ngf=64, n_blocks=6, use_cbam=True):
        super().__init__()
        # Initial convolution
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, kernel_size=7, stride=1, padding=0),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(True)
        ]

        # Downsampling
        in_features = ngf
        out_features = in_features * 2
        for _ in range(2):
            model += [
                nn.Conv2d(in_features, out_features, kernel_size=3, stride=2, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(True)
            ]
            in_features = out_features
            out_features = in_features * 2

        # ResNet blocks with CBAM attention
        for i in range(n_blocks):
            if use_cbam and i % 2 == 0:  # Apply CBAM every 2nd block
                model += [ResnetBlock(in_features, use_cbam=True)]
            else:
                model += [ResnetBlock(in_features, use_cbam=False)]

        # Upsampling
        out_features = in_features // 2
        for _ in range(2):
            model += [
                nn.ConvTranspose2d(in_features, out_features, kernel_size=3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(True)
            ]
            in_features = out_features
            out_features = in_features // 2

        # Output layer
        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, output_nc, kernel_size=7, stride=1, padding=0),
            nn.Tanh()
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


In [ ]:
import torch
import torch.nn as nn

# -------------------------------
# 🔹 Self-Attention Block (SAGAN style)
# -------------------------------
class SelfAttention(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.query_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_dim, in_dim, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))  # learnable weight

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        B, C, W, H = x.size()
        proj_query = self.query_conv(x).view(B, -1, W * H).permute(0, 2, 1)  # B × (N) × C'
        proj_key = self.key_conv(x).view(B, -1, W * H)  # B × C' × (N)
        energy = torch.bmm(proj_query, proj_key)  # B × N × N
        attention = self.softmax(energy)  # attention map
        proj_value = self.value_conv(x).view(B, -1, W * H)  # B × C × N

        out = torch.bmm(proj_value, attention.permute(0, 2, 1))  # B × C × N
        out = out.view(B, C, W, H)

        out = self.gamma * out + x  # Residual connection
        return out


# -------------------------------
# 🔹 PatchGAN Discriminator + Self-Attention
# -------------------------------
class NLayerDiscriminator(nn.Module):
    def __init__(self, input_nc=3, ndf=64, n_layers=3, use_attention=True):
        super().__init__()
        kw = 4
        padw = 1

        # First conv (no normalization here)
        sequence = [
            nn.Conv2d(input_nc, ndf, kernel_size=kw, stride=2, padding=padw),
            nn.LeakyReLU(0.2, True)
        ]

        nf_mult = 1
        nf_mult_prev = 1
        for n in range(1, n_layers):
            nf_mult_prev = nf_mult
            nf_mult = min(2 ** n, 8)
            sequence += [
                nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult,
                          kernel_size=kw, stride=2, padding=padw),
                nn.InstanceNorm2d(ndf * nf_mult),
                nn.LeakyReLU(0.2, True)
            ]

            # 🔹 Add Self-Attention after 2nd conv stage
            if use_attention and n == 2:
                sequence += [SelfAttention(ndf * nf_mult)]

        nf_mult_prev = nf_mult
        nf_mult = min(2 ** n_layers, 8)
        sequence += [
            nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult,
                      kernel_size=kw, stride=1, padding=padw),
            nn.InstanceNorm2d(ndf * nf_mult),
            nn.LeakyReLU(0.2, True)
        ]

        # Output layer (PatchGAN)
        sequence += [nn.Conv2d(ndf * nf_mult, 1, kernel_size=kw, stride=1, padding=padw)]

        self.model = nn.Sequential(*sequence)

    def forward(self, x):
        return self.model(x)


In [ ]:
# # Generators
# G_N2C = ResnetGenerator()
# G_C2N = ResnetGenerator()

# # Discriminators
# D_C = NLayerDiscriminator()
# D_N = NLayerDiscriminator()


In [ ]:
# Import and Locate files
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from itertools import cycle
from PIL import Image

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

WORK_DIR = "/kaggle/working"

# Find PNG folder
for root, dirs, files in os.walk("/kaggle/input"):
    pngs = [f for f in files if f.lower().endswith(".png")]
    if len(pngs) > 0:
        root_dir = root
        print("PNG root:", root_dir)
        print("Example:", pngs[:5])
        break

In [ ]:
# csv paths
train_csv = os.path.join(WORK_DIR, "CC_cancer_train.csv")
test_csv  = os.path.join(WORK_DIR, "CC_cancer_test.csv")

print("Train CSV exists:", os.path.exists(train_csv), train_csv)
print("Test CSV exists:", os.path.exists(test_csv), test_csv)
print("Image root exists:", os.path.exists(root_dir), root_dir)

In [ ]:
#Dataset classes
class SingleImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.samples = dataframe.to_dict(orient="records")
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row = self.samples[idx]

        image_path = os.path.join(
            self.root_dir,
            str(int(row["File Name"])) + ".png"
        )

        if not os.path.exists(image_path):
            return self.__getitem__((idx + 1) % len(self.samples))

        image = Image.open(image_path).convert("L")

        if str(row["Laterality"]).upper() == "R":
            image = image.transpose(Image.FLIP_LEFT_RIGHT)

        if self.transform:
            image = self.transform(image)

        return image

In [ ]:
#Transfors & datasets
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

def get_cycle_gan_datasets(csv_path, root_dir, transform):
    df_tmp = pd.read_csv(csv_path)

    df_C = df_tmp[df_tmp["density_domain"] == "C"].reset_index(drop=True)
    df_D = df_tmp[df_tmp["density_domain"] == "D"].reset_index(drop=True)

    dataset_C = SingleImageDataset(df_C, root_dir, transform)
    dataset_D = SingleImageDataset(df_D, root_dir, transform)

    return dataset_C, dataset_D

train_C, train_D = get_cycle_gan_datasets(train_csv, root_dir, train_transform)
test_C, test_D = get_cycle_gan_datasets(test_csv, root_dir, val_test_transform)

print("Train C:", len(train_C))
print("Train D:", len(train_D))
print("Test C:", len(test_C))
print("Test D:", len(test_D))

In [ ]:
#data loader
batch_size = 4

train_C_loader = DataLoader(
    train_C,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

sampler_D = WeightedRandomSampler(
    weights=[1.0] * len(train_D),
    num_samples=len(train_C),
    replacement=True
)

train_D_loader = DataLoader(
    train_D,
    batch_size=batch_size,
    sampler=sampler_D,
    num_workers=0
)

In [ ]:
ResnetGenerator
NLayerDiscriminator

In [ ]:
G_C2D = ResnetGenerator(input_nc=3, output_nc=3, n_blocks=6).to(device)
G_D2C = ResnetGenerator(input_nc=3, output_nc=3, n_blocks=6).to(device)

D_C = NLayerDiscriminator(input_nc=3, use_attention=True).to(device)
D_D = NLayerDiscriminator(input_nc=3, use_attention=True).to(device)

In [ ]:
#Losses & Generators
adversarial_loss = nn.MSELoss()
cycle_loss = nn.L1Loss()
identity_loss = nn.L1Loss()

lr = 2e-4
beta1 = 0.5

optimizer_G = optim.Adam(
    list(G_C2D.parameters()) + list(G_D2C.parameters()),
    lr=lr,
    betas=(beta1, 0.999)
)

optimizer_D_C = optim.Adam(D_C.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D_D = optim.Adam(D_D.parameters(), lr=lr, betas=(beta1, 0.999))

num_epochs = 50
lambda_cycle = 10.0
lambda_identity = 5.0

In [ ]:
#Training loop
for epoch in range(num_epochs):

    for i, (real_C, real_D) in enumerate(zip(train_C_loader, cycle(train_D_loader))):

        real_C = real_C.to(device)
        real_D = real_D.to(device)

        # -------------------------
        # Train Generators
        # -------------------------
        optimizer_G.zero_grad()

        fake_D = G_C2D(real_C)
        rec_C = G_D2C(fake_D)

        fake_C = G_D2C(real_D)
        rec_D = G_C2D(fake_C)

        idt_C = G_D2C(real_C)
        idt_D = G_C2D(real_D)

        loss_idt = identity_loss(idt_C, real_C) + identity_loss(idt_D, real_D)

        loss_GAN_C2D = adversarial_loss(D_D(fake_D), torch.ones_like(D_D(fake_D)))
        loss_GAN_D2C = adversarial_loss(D_C(fake_C), torch.ones_like(D_C(fake_C)))

        loss_cycle_C = cycle_loss(rec_C, real_C)
        loss_cycle_D = cycle_loss(rec_D, real_D)

        loss_G = (
            loss_GAN_C2D
            + loss_GAN_D2C
            + lambda_cycle * (loss_cycle_C + loss_cycle_D)
            + lambda_identity * loss_idt
        )

        loss_G.backward()
        optimizer_G.step()

        # -------------------------
        # Train Discriminator D_D
        # -------------------------
        optimizer_D_D.zero_grad()

        loss_D_real = adversarial_loss(D_D(real_D), torch.ones_like(D_D(real_D)))
        loss_D_fake = adversarial_loss(D_D(fake_D.detach()), torch.zeros_like(D_D(fake_D)))

        loss_D_D_total = 0.5 * (loss_D_real + loss_D_fake)

        loss_D_D_total.backward()
        optimizer_D_D.step()

        # -------------------------
        # Train Discriminator D_C
        # -------------------------
        optimizer_D_C.zero_grad()

        loss_D_real = adversarial_loss(D_C(real_C), torch.ones_like(D_C(real_C)))
        loss_D_fake = adversarial_loss(D_C(fake_C.detach()), torch.zeros_like(D_C(fake_C)))

        loss_D_C_total = 0.5 * (loss_D_real + loss_D_fake)

        loss_D_C_total.backward()
        optimizer_D_C.step()

        if i % 50 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}] "
                f"Iter [{i}] | "
                f"Loss_G: {loss_G.item():.4f} | "
                f"Loss_D_C: {loss_D_C_total.item():.4f} | "
                f"Loss_D_D: {loss_D_D_total.item():.4f}"
            )

    torch.save({
        "epoch": epoch + 1,
        "G_C2D": G_C2D.state_dict(),
        "G_D2C": G_D2C.state_dict(),
        "D_C": D_C.state_dict(),
        "D_D": D_D.state_dict(),
        "optimizer_G": optimizer_G.state_dict(),
        "optimizer_D_C": optimizer_D_C.state_dict(),
        "optimizer_D_D": optimizer_D_D.state_dict(),
    }, "/kaggle/working/cyclegan_density_checkpoint.pth")

    print("Checkpoint saved.")

In [ ]:
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# Get a batch from the dataloaders
normal_batch, _ = next(iter(test_normal_loader))
cancer_batch, _ = next(iter(test_cancer_loader))

# Move to device
normal_batch = normal_batch.to(device)
cancer_batch = cancer_batch.to(device)

# Generate images
with torch.no_grad():
    fake_C = G_N2C(normal_batch)  # Normal → Cancerous
    fake_N = G_C2N(cancer_batch)  # Cancerous → Normal

# Denormalize images from [-1,1] to [0,1] for visualization
def denorm(img_tensor):
    return (img_tensor + 1) / 2.0

fake_C = denorm(fake_C.cpu())
fake_N = denorm(fake_N.cpu())
normal_batch = denorm(normal_batch.cpu())
cancer_batch = denorm(cancer_batch.cpu())

# Plot a few examples
num_show = 4
fig, axes = plt.subplots(num_show, 4, figsize=(12, 8))

for i in range(num_show):
    axes[i,0].imshow(normal_batch[i].permute(1,2,0).squeeze(), cmap='gray')
    axes[i,0].set_title("Real Normal")
    axes[i,0].axis('off')
    
    axes[i,1].imshow(fake_C[i].permute(1,2,0).squeeze(), cmap='gray')
    axes[i,1].set_title("Fake Cancer")
    axes[i,1].axis('off')
    
    axes[i,2].imshow(cancer_batch[i].permute(1,2,0).squeeze(), cmap='gray')
    axes[i,2].set_title("Real Cancer")
    axes[i,2].axis('off')
    
    axes[i,3].imshow(fake_N[i].permute(1,2,0).squeeze(), cmap='gray')
    axes[i,3].set_title("Fake Normal")
    axes[i,3].axis('off')

plt.tight_layout()
plt.show()
